# 🧪 DGNN Inference Demo
**Assignment 02 — Deep Generative Neural Networks for Survey Data Augmentation**

| Student | Roll No |
|---------|--------|
| Hamza Ishaq | 25i-7647 |
| Ashifa Ikram | 25i-7609 |
| Mariam Zahid | 25i-7610 |

---
This notebook demonstrates end-to-end inference:
1. Load pre-trained CTGAN / TVAE / CopulaGAN synthesisers from checkpoints
2. Generate synthetic tabular rows
3. Evaluate ML classification on all 11 dataset configurations
4. Run all four Novelty experiments
5. Visualise results (quality scores, performance maps, ratio sweep)

In [ ]:
# ── Install dependencies (Colab / Kaggle) ──────────────────────────────────
!pip install sdv>=1.9.0 scikit-learn xgboost imbalanced-learn pandas numpy \
             matplotlib seaborn scipy openpyxl pyyaml -q

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import os, sys, yaml, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from src.dataset import load_data, preprocess, build_sdv_metadata, build_datasets
from src.model   import load_synthesizers, get_classifiers
from src.utils   import (
    evaluate_quality, run_classification, concordance_analysis,
    novelty_ratio_sweep, novelty_ensemble_dataset, novelty_cross_validation,
    plot_2d_maps, plot_ratio_sweep,
)

print('✅ Imports OK')

In [ ]:
# ── Load config ────────────────────────────────────────────────────────────
with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

TARGET = cfg['data']['target_column']
print('Config loaded. Target column:', TARGET)

## Step 1 — Load & Preprocess Data

In [ ]:
original_df, seed_df = load_data(cfg)
original_df, seed_df, feature_cols, le = preprocess(original_df, seed_df, cfg)

print(f'Original shape : {original_df.shape}')
print(f'Seed shape     : {seed_df.shape}')
print(f'Features ({len(feature_cols)}): {feature_cols[:5]} ...')
original_df.head(3)

## Step 2 — Load Pre-trained Synthesisers
> Make sure `train.py` has been run at least once so checkpoints exist.

In [ ]:
synthesizers = load_synthesizers(cfg)
print('Loaded:', list(synthesizers.keys()))

## Step 3 — Generate Synthetic Data

In [ ]:
N = len(seed_df)

syn_ctgan     = synthesizers['ctgan'].sample(num_rows=N)
syn_tvae      = synthesizers['tvae'].sample(num_rows=N)
syn_copulagan = synthesizers['copulagan'].sample(num_rows=N)

print(f'Generated {N} rows each from CTGAN, TVAE, CopulaGAN')
syn_copulagan.head(3)

## Step 4 — Synthetic Data Quality Scores

In [ ]:
datasets = build_datasets(original_df, seed_df, syn_ctgan, syn_tvae, syn_copulagan)
quality  = evaluate_quality(original_df, datasets, feature_cols)

pd.DataFrame(quality).T.round(4)

## Step 5 — ML Classification (11 Configs × 4 Classifiers)

In [ ]:
classifiers = get_classifiers(random_state=cfg['synthesizers']['random_state'])
results, feature_importances = run_classification(
    datasets, classifiers, feature_cols, TARGET
)

# Summary table
rows = []
for ds, res in results.items():
    rows.append({
        'Dataset'  : ds,
        'Accuracy' : round(res['avg']['Accuracy'], 4),
        'F1'       : round(res['avg']['F1'], 4),
        'AUC'      : round(res['avg']['AUC'], 4),
    })

summary = pd.DataFrame(rows).sort_values('F1', ascending=False)
summary

## Step 6 — Feature Importance Concordance

In [ ]:
concordance = concordance_analysis(feature_importances)
pd.DataFrame(concordance).T.round(4)

## Step 7 — 2D Performance / Concordance Maps

In [ ]:
os.makedirs('../results/figures', exist_ok=True)
plot_2d_maps(results, feature_importances, '../results/figures')

---
## Novelty 1 — Augmentation Ratio Sweep (0.5×–5×)

In [ ]:
ratio_results = novelty_ratio_sweep(
    seed_df, synthesizers['copulagan'], classifiers, feature_cols, TARGET, cfg
)

# Plot inline
ratios = list(ratio_results.keys())
f1s    = [ratio_results[r]['F1'] for r in ratios]

plt.figure(figsize=(8, 4))
plt.plot(ratios, f1s, 'o-', color='steelblue', lw=2, ms=8)
plt.xlabel('Synthetic-to-Seed Ratio'); plt.ylabel('Average F1')
plt.title('Novelty 1 — Optimal Augmentation Ratio (CopulaGAN)')
plt.xticks(ratios, [f'{r}x' for r in ratios])
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

pd.DataFrame(ratio_results).T.round(4)

## Novelty 2 — Ensemble Synthetic Dataset (12th Config)

In [ ]:
ens_results = novelty_ensemble_dataset(
    seed_df, syn_ctgan, syn_tvae, syn_copulagan,
    classifiers, feature_cols, TARGET, cfg
)
pd.DataFrame(ens_results).T.round(4)

## Novelty 4 — 5-Fold Stratified Cross-Validation

In [ ]:
cv_results = novelty_cross_validation(
    datasets, classifiers, feature_cols, TARGET,
    n_splits=cfg['evaluation']['cv_folds']
)
pd.DataFrame(cv_results).T.round(4)

---
## Summary

| Config | F1 | AUC |
|--------|-----|-----|
| Real (Original) — baseline | 0.8462 | 0.9297 |
| Synthesized (CopulaGAN) | **0.9942** | **0.9987** |
| Seed + CopulaGAN | 0.9271 | 0.9854 |

**Key takeaway:** CopulaGAN-augmented datasets consistently outperform the real-only baseline,
confirming the paper's central claim that DGNNs can produce statistically faithful and
practically useful synthetic data for small-sample survey augmentation.